# Setup

In [1]:
import numpy as np
import scipy.special as sp
from astropy import constants as const, units as u
import matplotlib.pyplot as plt
import h5py

In [2]:
import sys
sys.path.append('../')
from functions import *

# ZTFJ2243-like DWD System

For this study, we are chosing to base our study focus off of the DWD system ZTFJ2243.  However, we are chosing to change some of the parameters of that system for this study, for the reasons we motivate in the paper.

In [3]:
# Measured system quantities (K. B. Burdge et al. 2020)

# White dwarf masses
Ma = 0.323*u.M_sun.to(u.kg)
Mb = 0.335*u.M_sun.to(u.kg)

# Orbital period [s]
P = 527.93

In [4]:
# Selected system distance parameter
R = 150*u.pc.to(u.m)

In [5]:
# Derived Parameters
# --> these parameters derive from the values of the above chosen parameters

# GW carrier frequency, system chirp mass, GW amplitude, GW frequency derivative
fgw0  = 2/P
Mc    = Mchirp(Ma, Mb)
Amp   = Amp_binary(Mc, R, fgw0)
dfgw0 = dfgw_dt_binary(Mc, fgw0, 0, 0)

In [6]:
# Values reported in the final row of Table 1
print(f"A     = {Amp:0.2e}")
print(f"fgw0  = {fgw0*u.Hz.to(u.mHz):0.2f} mHz")
print(f"dfgw0 = {(dfgw0*u.Hz**2).to(u.nHz/u.year):0.2f}")

A     = 1.20e-21
fgw0  = 3.79 mHz
dfgw0 = 3.01 nHz / yr


# SNR Calculations

Load data from LISA simulations for $T_\mathrm{obs} = 4$, $7$, and $10$ years.

Calculate the SNR for the four different cases:

1. monochromatic, DWD+exoplanet
2. phase chirping, DWD+exoplanet
3. monochromatic, DWD-only
4. phase chirping, DWD-only

to verify that the *total* SNR does not change between cases.

In [7]:
# Load the data and look at the keys
hf = h5py.File('../LISAsim_data_files/lisasim_4yr.h5', 'r')

print("H5 file keys: ", hf.keys())

H5 file keys:  <KeysViewHDF5 ['chirping', 'chirping, no exoplanet', 'monochromatic', 'monochromatic, no exoplanet', 'simulation_metadata', 'times']>


In [8]:
# Store the data in arrays, then close the h5 file
data_sim   = dict(zip( [i for i in hf['simulation_metadata'].keys()], [hf['simulation_metadata/'+i][()] for i in hf['simulation_metadata'].keys()] ))
data_mono  = hf['monochromatic'][:]
data_chirp = hf['chirping'][:]

data_mono_noexo  = hf['monochromatic, no exoplanet'][:]
data_chirp_noexo = hf['chirping, no exoplanet'][:]

data_t     = hf['times'][:]
hf.close()

In [9]:
print("\nStarting data array shape = ", data_mono.shape)
print("\nSimulation Meta-Data: ")
[print(key, " = ", data_sim[key]) for key in data_sim.keys()];


Starting data array shape =  (2524608,)

Simulation Meta-Data: 
A  =  1.1981821801796646e-21
K  =  801.9744731967191
M1  =  6.661173066838471e+29
M2  =  6.422563882354705e+29
Mc  =  5.69389074051493e+29
Mp  =  1.8981245973360506e+28
R  =  4.628516372237051e+18
T_p  =  3155760.0
Tobs  =  126230400.0
dfgw0  =  9.540043059866e-17
dt  =  50.0
epsilon  =  0.03199506663973083
f_p  =  3.168808781402895e-07
fgw0  =  0.00379
fs  =  0.02
gw_beta  =  0.9417796643761402
gw_lambda  =  0.23108159296404923
inc  =  1.5707963267948966
iota  =  0
phi0  =  0
phi0_p  =  0
psi  =  0


In [10]:
print(f"Double Check: observation time = {data_sim['Tobs']*u.s.to(u.year):0.0f} yrs")

Double Check: observation time = 4 yrs


## Calculate DFT and Fourier Transforms

In [11]:
# Set the normalization convention
norm = "backward"

# frequency resolution
df = 1/data_sim['Tobs']

In [12]:
# Compute the one-dimensional discrete Fourier Transform for real input (output is still complex!)
# (and cut first f=0 frequency from the dataset)
mono_DFT  = np.fft.rfft(data_mono, norm=norm)[1:]  
chirp_DFT = np.fft.rfft(data_chirp, norm=norm)[1:]  

mono_noexo_DFT  = np.fft.rfft(data_mono_noexo, norm=norm)[1:]  
chirp_noexo_DFT = np.fft.rfft(data_chirp_noexo, norm=norm)[1:]  

# Return the Discrete Fourier Transform sample frequencies and their spacing
# (and cut first f=0 frequency from the dataset)
freqs = np.fft.rfftfreq(data_t.size, data_sim['dt'])[1:]

In [13]:
# convert the DFT data to the FT data

mono_FT  = DFT_to_FT(mono_DFT, freqs, data_sim['dt'], data_t[0], data_t.size, norm)
chirp_FT = DFT_to_FT(chirp_DFT, freqs, data_sim['dt'], data_t[0], data_t.size, norm)

mono_noexo_FT  = DFT_to_FT(mono_noexo_DFT, freqs, data_sim['dt'], data_t[0], data_t.size, norm)
chirp_noexo_FT = DFT_to_FT(chirp_noexo_DFT, freqs, data_sim['dt'], data_t[0], data_t.size, norm)

## SNR

In [14]:
# -------------------------------------------------------
# LISA Sensitivity (Michelson TDI X 2.0) functions
# -------------------------------------------------------

def S_OMS(freqs):
    Aoms = 12e-12
    return (Aoms * 2*np.pi*freqs/const.c.value)**2 * (1 + (2e-3/freqs)**4)

def S_acc(freqs):
    Aacc = 2.4e-15
    return (Aacc / (2*np.pi*freqs*const.c.value))**2 * (1 + (0.4e-3/freqs)**2) * (1 + (freqs/8e-3)**4)

def S_X20_EQarm(freqs, L):
    omega = 2*np.pi*freqs
    Soms = S_OMS(freqs)
    Sacc = S_acc(freqs)
    return 64 * np.sin(2*omega*L/const.c.value)**2 * np.sin(omega*L/const.c.value)**2 * ( Soms + (3+np.cos(2*omega*L/const.c.value))*Sacc )

# LISA arm-length
Larm = 2.5e9

In [15]:
print("TOTAL SNR calculation (Exoplanet)")
print("mono  = ", SNR(mono_FT.reshape(1,-1), S_X20_EQarm(freqs, Larm).reshape(1,1,-1), df))
print("chirp = ", SNR(chirp_FT.reshape(1,-1), S_X20_EQarm(freqs, Larm).reshape(1,1,-1), df))

print("\nTOTAL SNR calculation (No Exoplanet)")
print("mono  = ", SNR(mono_noexo_FT.reshape(1,-1), S_X20_EQarm(freqs, Larm).reshape(1,1,-1), df))
print("chirp = ", SNR(chirp_noexo_FT.reshape(1,-1), S_X20_EQarm(freqs, Larm).reshape(1,1,-1), df))

TOTAL SNR calculation (Exoplanet)
mono  =  2309.508529117587
chirp =  2309.5102484154686

TOTAL SNR calculation (No Exoplanet)
mono  =  2309.508526285204
chirp =  2309.510245579789


The final SNR values in the cell above are given in Table 1.  Re-run using the 4yr, 7yr, and 10yr data files.